# Lab 7: Fine-tuning Transformers

In the lecture, we saw that training Large Language Models (LLMs) from scratch is computationally prohibitive due to the massive number of parameters (e.g., GPT-3's 175B). We moved from the traditional ML paradigm to the LLM paradigm: pre-training (learning general knowledge) followed by post-training (adapting to specific tasks).

We specifically focused on LoRA (Low-Rank Adaptation). Instead of updating the full weight matrix $W$ during Supervised Fine-Tuning (SFT), LoRA freezes $W$ and injects trainable rank decomposition matrices $A$ and $B$ such that the forward pass becomes $h=W_0x +ΔWx=W_0x +BAx$ where $B \in \mathbf{R}^{dxr}, A \in \mathbf{R}^{rxk}$ and $r≪min(d,k)$.

In this lab, we calculate the number of trainable parameters necessary for LoRA and we observe how the rank $r$ and the position of LoRA modules in the pre-trained model affect the performance and the memory consumption.

In [ ]:
!pip install -q peft
!pip install -q datasets
!pip install bitsandbytes

In [ ]:
import torch
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, Trainer, DataCollatorForLanguageModeling, BitsAndBytesConfig
from huggingface_hub import login
from peft import LoraConfig, get_peft_model, TaskType
from datasets import load_dataset
import matplotlib.pyplot as plt
import numpy as np
import gc

# Check for GPU
device = "cuda" if torch.cuda.is_available() else "cpu"
print(f"Using device: {device}")



To use the Llama models from Hugging Face we need to create a token and accept the Llama license. Follow the steps below:
1. Accept the Llama license. Go to the model page: [meta-llama/Llama-3.2-1B](https://huggingface.co/meta-llama/Llama-3.2-1B) and accept the Community License Agreement. Note: Access is usually granted instantly. If it takes time, you might need to refresh the page after a few minutes.
2. Create a Hugging Face Access Token from https://huggingface.co/settings/tokens. Give the token a name like "LlamaLab". In the **Repositories Permissions** field add the meta-llama/Llama-3.2-1B and select the **Read access to contents of selected repos**. Click Create token. You will see a string starting with *hf_...* . Copy it immediately as you won't be able to see it again later.

Now you are ready to authenticate in HF with your token and use the Llama models.

In [ ]:
# HF authentication
hf_token = "" # TODO: Student adds token here
login(hf_token)

In [ ]:
# Load base model and tokenizer (a small model (1B parameters) is used to speed up the code)
model_name = "meta-llama/Llama-3.2-1B"
tokenizer = AutoTokenizer.from_pretrained(model_name)
base_model = AutoModelForCausalLM.from_pretrained(model_name)
tokenizer.pad_token = tokenizer.eos_token # Fix padding token issue

# Exercise 1: Calculate the number of LoRA parameters

The number of trainable parameters in LoRA is defined as:
$$
|\theta|=2 \cdot L_{LoRA} \cdot d_{model} \cdot r,
$$
where $L_{LoRA}$ is the number of layers we apply LoRA to and $d_{model}$ is the hidden model size.

**Tasks**:
1. Run the code below to apply LoRA with $r=8$ and print the trainable parameters.
2. Look at the model configuration (*base_model.config*) to find *num_hidden_layers* ($L_{LoRA}$) and *hidden_size* ($d_{model}$). Calculate the expected number of parameters using the formula above.
3.  Does your manual calculation match the trainable params printed by the library? If not, why might there be a discrepancy? (Hint: Check if bias terms are being trained).
4. Answer the question below.

In [ ]:
# Configure LoRA
lora_config = LoraConfig(
    r=8,
    lora_alpha=16,
    target_modules=["q_proj"], # we only target q_proj which is a square matrix (d_model x d_model) in Llama 3.2
    lora_dropout=0.05,
    bias="none",
    task_type=TaskType.CAUSAL_LM
)

# Get PEFT model
model = get_peft_model(base_model, lora_config)

# Print trainable parameters
model.print_trainable_parameters()

# --- TODO for STUDENTS ---
# TODO: Retrieve d_model and num_layers from config

# TODO for STUDENTS: Calculate expected params (2 * layers * d_model * r)
# Note: 'r' is 8

print(f"\n--- Manual Calculation ---")
print(f"Hidden Size (d_model): {d_model}")
print(f"Num Layers: {num_layers}")
print(f"Calculated Params: {expected_params:,}")
print(f"Actual Trainable Params: {model.num_parameters(only_trainable=True):,}")

####Question 1
The output shows a percentage (e.g., "trainable params: 0.5%"). Why does keeping this percentage low help preventing "catastrophic forgetting" of the pre-trained knowledge?

#### Answer 1

***Your answer goes here***

# Exercise 2: The role of rank ($r$)

Remember from the lecture that a small $r$ leads to fewer parameters and a larger $r$ allows to capture more task-specific information.

**Tasks**:
1. Train two models, one with $r=4$ and one with $r=64$ for one epoch on the "fka/awesome-chatgpt-prompts" dataset.
2. Compare the training loss. What can you conclude in terms of time required for training?
3. Is the memory used for training the two models significantly different? If you use Google Colab you can check that by opening the RAM and VRAM graphs in Colab (top right corner).
4. Answer the question below.

In [ ]:
# Load Data
dataset = load_dataset("fka/awesome-chatgpt-prompts")
train_sample = dataset["train"].select(range(50)) # Select a subset of the dataset for speed

def preprocess_function(examples):
    return tokenizer(examples["prompt"], truncation=True, padding="max_length", max_length=64)

tokenized_dataset = train_sample.map(preprocess_function, batched=True)
tokenized_dataset = tokenized_dataset.remove_columns(["act", "prompt"])

def train_model(model_obj, output_dir):
    training_args = TrainingArguments(
        output_dir=output_dir,
        per_device_train_batch_size=4,
        num_train_epochs=1,
        learning_rate=1e-3, # high lr for LoRA training
        logging_steps=10,
        report_to="none"
    )
    trainer = Trainer(
        model=model_obj,
        args=training_args,
        train_dataset=tokenized_dataset,
        data_collator=DataCollatorForLanguageModeling(tokenizer, mlm=False),
    )
    trainer.train()
    return trainer.state.log_history

In [ ]:
# --- TODO for STUDENTS ---
# Instantiate two models, one with r=4 and the other with r=64.
# Train both models (with one epochs) and keep track of the memory usage.
# Print the final training loss of the two models.

#### Question 2
If we increase $r$ to be equal to $d_{model}$, would LoRA become equivalent to FullFT? (Hint: Think about the matrix dimensions).

#### Answer 2

***Your answer goes here***

# Exercise 3: Where to apply LoRA

The original LoRA paper focuses on attention weights ($W_q, W_v$). However, recent work shows that applying LoRA also to FFN layers can improve the performance.

**Tasks**:
1. Configure two models. Model A applies LoRA only to attention matrices `["q_proj", "v_proj"]`. Model B applies LoRA also to the FFN layers `["q_proj", "v_proj", "gate_proj", "up_proj", "down_proj"]`.
2. How many more trainable parameters does Model B have compared to Model A?
3. The target_modules list requires specific string names matching the model architecture. Why can't we just use a generic name like "linear layer"? (Hint: Look at the model architecture using `print(base_model)`).
4. Answer the question below.

In [ ]:
# Scenario A: attention only
config_attn = LoraConfig(r=8, target_modules=["q_proj", "v_proj"], task_type=TaskType.CAUSAL_LM)
model_attn = get_peft_model(AutoModelForCausalLM.from_pretrained(model_name), config_attn)

# Scenario B: attention + FFN
config_all = LoraConfig(
    r=8,
    target_modules=["q_proj", "v_proj", "gate_proj", "up_proj", "down_proj"],
    task_type=TaskType.CAUSAL_LM
)
model_all = get_peft_model(AutoModelForCausalLM.from_pretrained(model_name), config_all)

print("--- Attention Only ---")
model_attn.print_trainable_parameters()

print("\n--- Attention + FFN ---")
model_all.print_trainable_parameters()

# TODO for STUDENTS
# Calculate the percentage increase in trainable parameters when adding FFN


# TODO for STUDENTS
# Inspect the model architecture

#### Question 3

If you are constrained by GPU memory but need maximum performance, would you choose a high $r$ but apply LoRAs only to the attention matrices or use a low $r$ on attention + FFN matrices? Why? (Hint: Think about the recommendations in the lecture).

#### Answer 3

***Your answer goes here***

# Bonus exercise: QLoRA (Quantized LoRA)

QLoRA quantizes the base model to 4-bit precision while keeping LoRA adapters in full precision. During the training process, the compressed weights are temporarily dequantized back to 16-bit precision only for the forward and backward passes to ensure calculation accuracy. This allows users to perform high-precision updates on the small LoRA adapters while keeping the heavy base model in a highly efficient, compressed state.

**Tasks**:
1. Load the base model in 4-bit precision using the `BitsAndBytesConfig` below.
2. Compare the memory footprint of the QLoRA model vs. the standard 16-bit model used in previous exercises. If you are using Google Colab, you can monitor the 'VRAM' bar in Colab to see the difference visually.
3. Answer the question below.

In [ ]:
# Define the quantization config
bnb_config = BitsAndBytesConfig(
    load_in_4bit=True,
    bnb_4bit_compute_dtype=torch.bfloat16,
    bnb_4bit_use_double_quant=True,
)

# Load in 4-bit
model_4bit = AutoModelForCausalLM.from_pretrained(
    model_name,
    quantization_config=bnb_config,
    device_map="auto"
)

# TODO for STUDENTS
# Check memory usage. You can also monitor the 'VRAM' bar in Colab to see the difference visually!

#### Question 4

In QLoRA, we store the massive base model in 4-bit to save memory, but we keep the small LoRA adapters in 16-bit. Why don't we also compress the LoRA adapters to 4-bit to save even more memory? (Hint: Think about what we actually train).

#### Answer 4

***Your answer goes here***